In [1]:
# Load libraries and set paths 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

SEED = 12345
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

DATA = Path("mini-atlas/data")
QC = Path("results/QC")
MORPH = Path("results/morphology")
OUT = Path("results/models")
FIGS = Path("results/models/figures")
OUT.mkdir(parents=True, exist_ok=True)
FIGS.mkdir(parents=True, exist_ok=True)

# 1. Load data 

In [2]:
# Load covariates 
covars = pd.read_csv(QC / "covars.csv", index_col=0, dtype={"batch": str})   # Batch kept as text
print("covariates:", covars.shape)
covars.head(3)

covariates: (605, 7)


,mouse,batch,rna_family,log10_ngenes,intron_frac,norm_depth,exon_reads
20180306_sample_2,mouse_YMIOM,2,IT,3.767379,0.586967,0.175714,204061
20190418_sample_4,mouse_HZREI,7,CT,3.714414,0.244747,0.819682,2365129
20190926_sample_2,mouse_BSSDD,10,ET,4.022717,0.543314,0.512602,5941397


In [3]:
# Rebuild expression from raw counts 

exon = pd.read_csv(DATA / "m1_patchseq_exon_counts.csv.gz", index_col=0)

counts = exon[covars.index].T                       # cells x genes, same cells and order as covars
lib = counts.sum(axis=1)
expr = np.log2(counts.div(lib, axis=0) * 1e6 + 1)   # log2(CPM + 1), same as notebook 01
detected = counts > 0

print("expression:", expr.shape, "| any NaN:", bool(expr.isna().any().any()))

expression: (605, 42466) | any NaN: False


In [4]:
# Load morphology axes and align cells 

pcs = pd.read_csv(MORPH / "morph_pcs.csv", index_col=0)
print("axes:", pcs.shape, "|", pcs.columns.tolist())

cells = covars.index.intersection(pcs.index)
covars, pcs = covars.loc[cells], pcs.loc[cells]
expr, detected = expr.loc[cells], detected.loc[cells]
print("cells in the analysis:", len(cells))
print(covars["rna_family"].value_counts().to_dict())

axes: (603, 8) | ['wPC1', 'wPC2', 'wPC3', 'wPC4', 'gPC1', 'gPC2', 'gPC3', 'gPC4']
cells in the analysis: 603
{'IT': 150, 'Pvalb': 142, 'Sst': 101, 'CT': 67, 'Vip': 59, 'Lamp5': 45, 'ET': 33, 'Sncg': 6}


# 2. Choose the cell families to model 

In [5]:
# Group data by RNA family and check the number of mice and corresponding batch 
fam_n = (covars.groupby("rna_family")
         .agg(cells=("mouse", "size"), mice=("mouse", "nunique"), batches=("batch", "nunique"))
         .sort_values("cells", ascending=False))
print(fam_n.to_string())

            cells  mice  batches
rna_family                      
IT            150    91       12
Pvalb         142    79       10
Sst           101    47        8
CT             67    44        6
Vip            59    33        7
Lamp5          45    37        7
ET             33    23        7
Sncg            6     6        5


In [6]:
# Set minimum cell count 
MIN_CELLS = 30 

# Select families meeting that threshold 
strata = fam_n.index[fam_n["cells"] >= MIN_CELLS].tolist()
print("\nfamilies modelled on their own (>= {} cells):".format(MIN_CELLS), strata)
print("cells in modelled families:", int(fam_n.loc[strata, "cells"].sum()), "of", len(covars))


families modelled on their own (>= 30 cells): ['IT', 'Pvalb', 'Sst', 'CT', 'Vip', 'Lamp5', 'ET']
cells in modelled families: 597 of 603


# 3. Filter genes to model 

In [7]:
MIN_DET = 0.20      # gene must be detected (count > 0) in at least 20% of a family's cells

passes = {}
for s in strata:
    idx = covars.index[covars["rna_family"] == s]
    passes[s] = detected.loc[idx].mean() >= MIN_DET
passes = pd.DataFrame(passes)

print("genes passing the filter, per family:")
print(passes.sum().to_string())

n_pass = passes.sum(axis=1)
universe = n_pass.index[n_pass >= 2]
print("\ngenes passing in at least 2 families (tested):", len(universe))

genes passing the filter, per family:
IT       13201
Pvalb    12152
Sst      11397
CT       11368
Vip      10998
Lamp5    11417
ET       15928

genes passing in at least 2 families (tested): 13934


# 4. Model design 

In [8]:
# Create a design matrix for one RNA family
def design(cov, min_batch=5):
    """Build covariates for one RNA family."""

    # Add an intercept (baseline)
    X = pd.DataFrame({"intercept": 1.0}, index=cov.index)

    # Standardise continuous technical covariates
    for c in ["log10_ngenes", "intron_frac", "norm_depth"]:
        v = cov[c]
        X[c] = (v - v.mean()) / v.std(ddof=0)

    # Combine batches with fewer than min_batch cells into a "rare" category 
    b = cov["batch"].where(
        cov["batch"].map(cov["batch"].value_counts()) >= min_batch,
        "rare"
    )

    # Encode batches as dummy variables, omitting the reference category
    X = pd.concat([
        X,
        pd.get_dummies(b, prefix="batch", drop_first=True, dtype=float)
    ], axis=1)

    # Return the design matrix
    return X

In [9]:
# Run the function 
rows = []
for s in strata:
    cov = covars[covars["rna_family"] == s]
    X = design(cov)
    rows.append({"family": s, "cells": len(cov), "mice": cov["mouse"].nunique(),
                 "parameters": X.shape[1], "full_rank": np.linalg.matrix_rank(X.values) == X.shape[1]})
print(pd.DataFrame(rows).to_string(index=False))

family  cells  mice  parameters  full_rank
    IT    150    91          14       True
 Pvalb    142    79          13       True
   Sst    101    47          11       True
    CT     67    44           7       True
   Vip     59    33           7       True
 Lamp5     45    37           7       True
    ET     33    23           6       True


# 5. Fit model 

In [10]:
def fit_family(Y, X, x, clusters):
    """Slope of every gene (columns of Y) on axis x, adjusting for covariates X,
    with standard errors clustered by mouse (CR1)."""
    Q, _ = np.linalg.qr(X)                       # orthonormal basis of the covariates
    Yr = Y - Q @ (Q.T @ Y)                       # expression with covariates removed
    xr = x - Q @ (Q.T @ x)                       # axis with covariates removed
    sxx = xr @ xr
    beta = xr @ Yr / sxx                         # slope per gene
    resid = Yr - np.outer(xr, beta)

    order = np.argsort(clusters, kind="stable")
    starts = np.r_[0, np.flatnonzero(np.diff(clusters[order])) + 1]
    scores = np.add.reduceat((xr[:, None] * resid)[order], starts, axis=0)   # summed within each mouse
    G, n, k = len(starts), X.shape[0], X.shape[1]
    correction = G / (G - 1) * (n - 1) / (n - k - 1)
    se = np.sqrt(correction * (scores ** 2).sum(0)) / abs(sxx)
    return beta, se

In [11]:
# Test against standard library 

import statsmodels.api as sm

s = "IT"
cov = covars[covars["rna_family"] == s]
X = design(cov).values
genes_s = passes.index[passes[s]]
Y = expr.loc[cov.index, genes_s].values
x = pcs.loc[cov.index, "wPC1"].values
codes = pd.factorize(cov["mouse"])[0]

beta, se = fit_family(Y, X, x, codes)

for g in [0, 500, 5000]:
    ref = sm.OLS(Y[:, g], np.column_stack([x, X])).fit(
        cov_type="cluster", cov_kwds={"groups": codes, "use_correction": True, "df_correction": True})
    print(genes_s[g], "| mine:", round(beta[g], 5), round(se[g], 5),
          "| statsmodels:", round(ref.params[0], 5), round(ref.bse[0], 5))

0610009B22Rik | mine: -0.09729 0.41539 | statsmodels: -0.09729 0.41539
Adck2 | mine: 0.29249 0.34206 | statsmodels: 0.29249 0.34206
Gm45734 | mine: -0.14748 0.1369 | statsmodels: -0.14748 0.1369


# 6. Fit all families and axes 

In [12]:
axes = ["wPC1", "wPC2", "wPC3", "wPC4"]
beta = {a: pd.DataFrame(np.nan, index=universe, columns=strata) for a in axes}
se   = {a: pd.DataFrame(np.nan, index=universe, columns=strata) for a in axes}

for s in strata:
    cov = covars[covars["rna_family"] == s]
    X = design(cov).values
    genes_s = universe[passes.loc[universe, s].values]       # tested genes that pass the filter in this family
    Y = expr.loc[cov.index, genes_s].values
    codes = pd.factorize(cov["mouse"])[0]
    for a in axes:
        b, e = fit_family(Y, X, pcs.loc[cov.index, a].values, codes)
        beta[a].loc[genes_s, s] = b
        se[a].loc[genes_s, s] = e

print("gene estimates per family (wPC1):")
print(beta["wPC1"].notna().sum().to_string())
print("\ntotal estimates:", sum(int(beta[a].notna().sum().sum()) for a in axes))
print("smallest standard error:", min(float(se[a].min().min()) for a in axes))

gene estimates per family (wPC1):
IT       13026
Pvalb    12052
Sst      11348
CT       11316
Vip      10921
Lamp5    11336
ET       13446

total estimates: 333780
smallest standard error: 0.014959959529705523


# 7. Meta-analysis across families and FDR 

In [13]:
def meta(beta_df, se_df):
    B, S = beta_df.values, se_df.values
    w = np.where(np.isfinite(S), 1 / S ** 2, 0.0)            # precision weights, 0 where the gene wasn't fitted
    n_fam = (w > 0).sum(1)
    sw = w.sum(1)
    est = (w * np.nan_to_num(B)).sum(1) / sw
    se_m = 1 / np.sqrt(sw)
    Q = (w * (np.nan_to_num(B) - est[:, None]) ** 2).sum(1)   # Cochran's Q
    het_p = stats.chi2.sf(Q, np.maximum(n_fam - 1, 1))
    z = est / se_m
    return pd.DataFrame({"beta": est, "se": se_m, "z": z, "n_families": n_fam, "het_p": het_p},
                        index=beta_df.index)

res = []
for a in axes:
    r = meta(beta[a], se[a])
    r["axis"] = a
    r = pd.concat([r, beta[a].add_prefix("beta_")], axis=1)
    res.append(r)
res = pd.concat(res).rename_axis("gene").reset_index()

res["p"] = 2 * stats.norm.sf(res["z"].abs())
res["q"] = stats.false_discovery_control(res["p"].values, method="bh")   # BH across all gene x axis tests

lam = np.median(res["z"] ** 2) / stats.chi2.ppf(0.5, 1)
print("tests:", len(res), "| genomic inflation (lambda):", round(lam, 2))
print("\ngenes with q < 0.05, per axis:")
print(res[res["q"] < 0.05].groupby("axis")["gene"].nunique().to_string())
print("\ntop 10 by p-value:")
print(res.sort_values("p").head(10)[["gene", "axis", "beta", "z", "p", "q", "n_families", "het_p"]].round(4).to_string(index=False))

# Save file 
res.to_csv(OUT / "meta_results.csv", index=False)

tests: 55736 | genomic inflation (lambda): 1.26

genes with q < 0.05, per axis:
axis
wPC1    2
wPC2    7
wPC3    1
wPC4    5

top 10 by p-value:
   gene axis    beta       z   p      q  n_families  het_p
  Aggf1 wPC4  0.6484  5.8518 0.0 0.0003           7 0.0000
   Spen wPC4  0.5335  5.2644 0.0 0.0039           7 0.0003
   Ccn2 wPC2  0.9575  5.1267 0.0 0.0046           3 0.2552
 Gm6563 wPC4 -0.4404 -5.1040 0.0 0.0046           7 0.0000
    Kit wPC4  1.6648  4.9983 0.0 0.0064           2 0.3000
  Prdx2 wPC2 -0.5709 -4.8010 0.0 0.0147           7 0.0170
Gm11989 wPC1 -0.1052 -4.6676 0.0 0.0243           7 0.7077
Gm12231 wPC3  0.3791  4.5900 0.0 0.0309           7 0.0023
  Nxph4 wPC2  0.9251  4.5526 0.0 0.0328           2 0.8665
  Foxp2 wPC2 -1.3218 -4.5076 0.0 0.0330           2 0.2715


In [14]:
# Check if the gene hits are consistent across families 

bcols = [f"beta_{s}" for s in strata]
hits = res[res["q"] < 0.05].sort_values("p").copy()
b = hits[bcols]
hits["same_sign"] = (np.sign(b).eq(np.sign(hits["beta"]), axis=0) & b.notna()).sum(axis=1)

print(hits[["gene", "axis", "beta", "q", "n_families", "same_sign", "het_p"] + bcols].round(2).to_string(index=False))

         gene axis  beta    q  n_families  same_sign  het_p  beta_IT  beta_Pvalb  beta_Sst  beta_CT  beta_Vip  beta_Lamp5  beta_ET
        Aggf1 wPC4  0.65 0.00           7          4   0.00     0.37       -0.57     -0.20     0.48     -0.28        0.48     2.07
         Spen wPC4  0.53 0.00           7          5   0.00     0.35        0.30      0.74     0.27      1.32       -0.64    -0.16
         Ccn2 wPC2  0.96 0.00           3          3   0.26      NaN         NaN       NaN     1.26      0.71         NaN     0.31
       Gm6563 wPC4 -0.44 0.00           7          6   0.00     0.17       -0.06     -0.31    -1.06     -0.77       -0.78    -0.08
          Kit wPC4  1.66 0.01           2          2   0.30      NaN         NaN       NaN      NaN      1.86        1.07      NaN
        Prdx2 wPC2 -0.57 0.01           7          5   0.02     0.06       -0.51     -0.06    -1.08     -0.77       -0.70     0.34
      Gm11989 wPC1 -0.11 0.02           7          6   0.71    -0.16       -0.08   

In [15]:
# Save gene hits file 

hits.to_csv(OUT / "hits_per_family.csv")